In [ ]:
import pandas as pd

#config
muller_file = "../../raw/muller_data.txt"
output_file = "../../processed/muller_data.csv"

In [2]:
#load
with open(muller_file, "r") as f:
    first_line = f.readline()
sep = "\t" if "\t" in first_line else ","

muller = pd.read_csv(muller_file, sep=sep, dtype=str)
muller.head()

,patient,dataset,train_test,response_type,Nb_Samples,Sample_Tissue,Cancer_Type,chromosome,genomic_coord,ref,...,bestMutationScore_I,bestWTPeptideCount_I,bestWTMatchType_I,CSCAPE_score,Clonality,CCF,nb_same_mutation_Intogen,mutation_driver_statement_Intogen,gene_driver_Intogen,seq_len
0,4278,NCI,train,negative,NaN,Colon,Colon Adenocarcinoma,16,19718494.0,T,...,1.0,26,EXACT,0.708244,clonal,0.8768072885719944,NaN,NaN,NaN,9
1,4278,NCI,train,negative,NaN,Colon,Colon Adenocarcinoma,6,13327094.0,A,...,3.0,27,INCLUDED,0.865282,clonal,0.7432528409090909,0.0,NaN,NaN,9
2,4278,NCI,train,negative,NaN,Colon,Colon Adenocarcinoma,8,117862919.0,C,...,0.0,92,NONE,0.805806,clonal,1.0879345603271984,0.0,NaN,OTHER TUMOR DRIVER,9
3,4278,NCI,train,CD8,NaN,Colon,Colon Adenocarcinoma,15,49421687.0,G,...,3.0,67,PARTIAL_MUT,0.927601,clonal,0.927643784786642,0.0,NaN,OTHER TUMOR DRIVER,9
4,4278,NCI,train,negative,NaN,Colon,Colon Adenocarcinoma,15,49421687.0,G,...,3.0,67,PARTIAL_MUT,0.927601,clonal,0.927643784786642,0.0,NaN,OTHER TUMOR DRIVER,8


In [3]:
#PROCESS MULLER
print(f"raw rows: {len(muller)}")
print(muller.columns) 
print(muller['mutation_type'].unique())

raw rows: 1787710
Index(['patient', 'dataset', 'train_test', 'response_type', 'Nb_Samples',
       'Sample_Tissue', 'Cancer_Type', 'chromosome', 'genomic_coord', 'ref',
       'alt', 'gene', 'protein_coord', 'aa_mutant', 'aa_wt', 'mutant_seq',
       'wt_seq', 'pep_mut_start', 'TumorContent', 'mutation_type', 'Zygosity',
       'mutant_best_alleles', 'wt_best_alleles',
       'mutant_best_alleles_netMHCpan',
       'mutant_other_significant_alleles_netMHCpan',
       'wt_best_alleles_netMHCpan', 'mutant_rank', 'mutant_rank_netMHCpan',
       'mutant_rank_PRIME', 'mut_Rank_Stab', 'TAP_score',
       'mut_netchop_score_ct', 'mut_binding_score', 'mut_is_binding_pos',
       'mut_aa_coeff', 'DAI_NetMHC', 'DAI_MixMHC', 'DAI_NetStab',
       'mutant_other_significant_alleles', 'DAI_MixMHC_mbp', 'rnaseq_TPM',
       'rnaseq_alt_support', 'GTEx_all_tissues_expression_mean',
       'Sample_Tissue_expression_GTEx', 'TCGA_Cancer_expression',
       'bestWTMatchScore_I', 'bestWTMatchOverlap_I', 'b

In [4]:
muller['response_type'].unique()

array(['negative', 'CD8', 'not_tested'], dtype=object)

In [9]:
#same label
response_map = {
    'CD8': '1', 
    'negative': '0'
}

In [10]:
#keep only tested
print(f"before filtering for tested rows: {len(muller)}")
prime = pd.DataFrame() 
prime = muller[muller['response_type'].isin(response_map.keys())].copy()
print(f"after filtering for tested rows: {len(prime)}")

before filtering for tested rows: 1787710
after filtering for tested rows: 423085


In [11]:
#clean seq
prime["mt_seq"] = muller["mutant_seq"].str.strip().str.upper()
prime["wt_seq"] = muller["wt_seq"].str.strip().str.upper()

#drop rows with no peptide seq
prime = prime.dropna(subset=["mt_seq", "wt_seq"]) 
print("after removing empty rows:", len(prime))

after removing empty rows: 423085


In [12]:
#filter out invalid aa
valid_aa = set("ACDEFGHIKLMNPQRSTVWY")
def is_valid(seq): 
    if not isinstance(seq, str): 
        return False
    for aa in seq:
        if aa not in valid_aa: 
            return False
    return True
mask = prime["mt_seq"].apply(is_valid)
prime = prime[mask]
mask2 = prime["wt_seq"].apply(is_valid)
prime = prime[mask2]
print("after filtering invalid aa:", len(prime))

after filtering invalid aa: 423085


In [13]:
#convert labels
prime["label"] = muller["response_type"].map(response_map)
prime.head() 

,patient,dataset,train_test,response_type,Nb_Samples,Sample_Tissue,Cancer_Type,chromosome,genomic_coord,ref,...,bestWTMatchType_I,CSCAPE_score,Clonality,CCF,nb_same_mutation_Intogen,mutation_driver_statement_Intogen,gene_driver_Intogen,seq_len,mt_seq,label
0,4278,NCI,train,negative,NaN,Colon,Colon Adenocarcinoma,16,19718494.0,T,...,EXACT,0.708244,clonal,0.8768072885719944,NaN,NaN,NaN,9,DPKLKFLRL,0
1,4278,NCI,train,negative,NaN,Colon,Colon Adenocarcinoma,6,13327094.0,A,...,INCLUDED,0.865282,clonal,0.7432528409090909,0.0,NaN,NaN,9,QRNFRSVHY,0
2,4278,NCI,train,negative,NaN,Colon,Colon Adenocarcinoma,8,117862919.0,C,...,NONE,0.805806,clonal,1.0879345603271984,0.0,NaN,OTHER TUMOR DRIVER,9,QLIPELKLL,0
3,4278,NCI,train,CD8,NaN,Colon,Colon Adenocarcinoma,15,49421687.0,G,...,PARTIAL_MUT,0.927601,clonal,0.927643784786642,0.0,NaN,OTHER TUMOR DRIVER,9,KPYTRIHIL,1
4,4278,NCI,train,negative,NaN,Colon,Colon Adenocarcinoma,15,49421687.0,G,...,PARTIAL_MUT,0.927601,clonal,0.927643784786642,0.0,NaN,OTHER TUMOR DRIVER,8,YTRIHILF,0


In [14]:
#match cedar columns
prime["mutation"] = muller["aa_mutant"].fillna("")
prime["epitope_relation"] = muller["mutation_type"].fillna("")
prime["source_molecule"] = muller["gene"].fillna("")
prime["data_source"] = "Muller2023"
prime.head() 

,patient,dataset,train_test,response_type,Nb_Samples,Sample_Tissue,Cancer_Type,chromosome,genomic_coord,ref,...,nb_same_mutation_Intogen,mutation_driver_statement_Intogen,gene_driver_Intogen,seq_len,mt_seq,label,mutation,epitope_relation,source_molecule,data_source
0,4278,NCI,train,negative,NaN,Colon,Colon Adenocarcinoma,16,19718494.0,T,...,NaN,NaN,NaN,9,DPKLKFLRL,0,P,SNV,KNOP1,Muller2023
1,4278,NCI,train,negative,NaN,Colon,Colon Adenocarcinoma,6,13327094.0,A,...,0.0,NaN,NaN,9,QRNFRSVHY,0,H,SNV,TBC1D7,Muller2023
2,4278,NCI,train,negative,NaN,Colon,Colon Adenocarcinoma,8,117862919.0,C,...,0.0,NaN,OTHER TUMOR DRIVER,9,QLIPELKLL,0,K,SNV,RAD21,Muller2023
3,4278,NCI,train,CD8,NaN,Colon,Colon Adenocarcinoma,15,49421687.0,G,...,0.0,NaN,OTHER TUMOR DRIVER,9,KPYTRIHIL,1,L,SNV,COPS2,Muller2023
4,4278,NCI,train,negative,NaN,Colon,Colon Adenocarcinoma,15,49421687.0,G,...,0.0,NaN,OTHER TUMOR DRIVER,8,YTRIHILF,0,L,SNV,COPS2,Muller2023


In [15]:
prime["label"].value_counts()

label
0    422907
1       178
Name: count, dtype: int64

In [16]:
prime = prime[["mt_seq", "wt_seq", "label", "mutation", "epitope_relation", "source_molecule", "data_source"]]
prime.head()

,mt_seq,wt_seq,label,mutation,epitope_relation,source_molecule,data_source
0,DPKLKFLRL,DQKLKFLRL,0,P,SNV,KNOP1,Muller2023
1,QRNFRSVHY,QRNFRSVYY,0,H,SNV,TBC1D7,Muller2023
2,QLIPELKLL,QLIPELELL,0,K,SNV,RAD21,Muller2023
3,KPYTRIHIL,KPYTRIHIP,1,L,SNV,COPS2,Muller2023
4,YTRIHILF,YTRIHIPF,0,L,SNV,COPS2,Muller2023


In [17]:
print(len(prime))
print(prime["label"].value_counts())

423085
label
0    422907
1       178
Name: count, dtype: int64


In [18]:
#saving output file
prime.to_csv(output_file, index=False)